# DCMTK sample worklists — real test fixtures

This notebook fetches **real DICOM Modality Worklist test fixtures** from the OFFIS DCMTK project — the standard reference DICOM toolkit, peer-reviewed and used by virtually every DICOM project worldwide.

> Important naming note: I previously referred to "dcm4che" — that was wrong. The toolkit that ships canonical MWL test fixtures is **DCMTK** (OFFIS DICOM Toolkit), a separate project from dcm4che. DCMTK is what we actually want.

What this notebook does:

1. Sparse-clones the DCMTK repository (only the worklist data subdirectory, ~few MB)
2. Lists the `.dump` test fixtures shipped in `dcmwlm/data/wlistdb/OFFIS/`
3. Converts each `.dump` file to a binary `.wl` file using a Python parser
4. Reads each `.wl` back with pydicom and shows what's inside
5. Surveys priorities, modalities, and patient demographics across all fixtures
6. Provides instructions for copying them into the CritCom repo

**Citation for the paper:**

> Eichelberg M, Riesmeier J, Wilkens T, Hewett AJ, Barth A, Jensch P. *Ten years of medical imaging standardization and prototypical implementation: the DICOM standard and the OFFIS DICOM toolkit (DCMTK)*. Proceedings of SPIE Medical Imaging. 2004;5371:57-68.

> "DICOM Modality Worklist test fixtures were sourced from the OFFIS DCMTK reference implementation (`github.com/DCMTK/dcmtk`, commit \<sha\>), specifically the `dcmwlm/data/wlistdb/OFFIS/` test database that ships with the toolkit."

---

### Requirements

```bash
pip install pydicom
```

Plus `git` available on the command line (you used it to clone Synthea earlier — same setup).


## 1. Setup

In [ ]:
import os
import subprocess
from pathlib import Path

WORK_DIR = Path.cwd() / "dcmtk_workspace"
DCMTK_DIR = WORK_DIR / "dcmtk"
WL_DIR = WORK_DIR / "worklists"
DUMP_DIR = WORK_DIR / "dumps"

WORK_DIR.mkdir(exist_ok=True)
WL_DIR.mkdir(exist_ok=True)
DUMP_DIR.mkdir(exist_ok=True)

print(f"Working in: {WORK_DIR}")

In [ ]:
# Verify pydicom is available
try:
    import pydicom
    print(f"pydicom OK — version {pydicom.__version__}")
except ImportError:
    print("Install pydicom first:  pip install pydicom matplotlib")

## 2. Sparse-clone DCMTK (just the worklist data)

DCMTK is a fairly large repo. We only need `dcmwlm/data/wlistdb/OFFIS/`, so we use git's sparse-checkout to fetch just that subdirectory.

In [ ]:
# Sparse-clone DCMTK — fetches only the directories we need
if not DCMTK_DIR.exists():
    print("Sparse-cloning DCMTK (only dcmwlm/data subdirectory)...")
    DCMTK_DIR.mkdir()
    subprocess.run(
        ["git", "init"],
        cwd=str(DCMTK_DIR), check=True, capture_output=True,
    )
    subprocess.run(
        ["git", "remote", "add", "origin", "https://github.com/DCMTK/dcmtk.git"],
        cwd=str(DCMTK_DIR), check=True, capture_output=True,
    )
    subprocess.run(
        ["git", "config", "core.sparseCheckout", "true"],
        cwd=str(DCMTK_DIR), check=True, capture_output=True,
    )
    sparse_file = DCMTK_DIR / ".git" / "info" / "sparse-checkout"
    sparse_file.write_text("dcmwlm/data/\n")
    subprocess.run(
        ["git", "pull", "--depth=1", "origin", "master"],
        cwd=str(DCMTK_DIR), check=True, capture_output=True,
    )
    print("Done.")
else:
    print("DCMTK already cloned.")

# Show what we got
worklist_data_dir = DCMTK_DIR / "dcmwlm" / "data" / "wlistdb" / "OFFIS"
if worklist_data_dir.exists():
    print(f"\nWorklist database directory: {worklist_data_dir}")
    print("Files:")
    for f in sorted(worklist_data_dir.iterdir()):
        print(f"  {f.name}  ({f.stat().st_size} bytes)")
else:
    print("ERROR: expected directory not found — DCMTK layout may have changed.")
    print(f"Looking under {DCMTK_DIR / 'dcmwlm' / 'data'} for any data files:")
    data_dir = DCMTK_DIR / "dcmwlm" / "data"
    if data_dir.exists():
        for p in data_dir.rglob("*"):
            if p.is_file():
                print(f"  {p.relative_to(DCMTK_DIR)}")

## 3. Inspect a `.dump` fixture

Each fixture is a text-format DICOM dump. Let's look at one to see the data DCMTK ships.

In [ ]:
worklist_data_dir = DCMTK_DIR / "dcmwlm" / "data" / "wlistdb" / "OFFIS"
dump_files = sorted(worklist_data_dir.glob("*.dump"))
print(f"Found {len(dump_files)} .dump fixtures")

if dump_files:
    sample = dump_files[0]
    print(f"\n=== {sample.name} ===")
    print(sample.read_text())

## 4. Convert `.dump` to binary `.wl`

DCMTK ships these as `.dump` text files because they're more readable / editable. To use them with Orthanc (or any other MWL SCP), we need binary `.wl` files. DCMTK has a tool called `dump2dcm` for this conversion, but installing the full DCMTK build is heavy.

Instead we parse the dump format ourselves with a small Python helper and re-emit as DICOM using pydicom. The dump format is straightforward — each line is `(group,element) VR [value]` or `(group,element) VR ; comment`.

In [ ]:
import re
from pydicom.dataset import Dataset, FileMetaDataset
from pydicom.uid import ExplicitVRLittleEndian, generate_uid

# Regex to parse a single line of a DCMTK dump file.
# The real format is:  (gggg,eeee) VR  VALUE
# where VALUE may be plain text, optionally wrapped in [], or empty.
LINE_RE = re.compile(
    r"\s*\(([0-9a-fA-F]{4}),([0-9a-fA-F]{4})\)\s+(\w{2})\s*(.*?)\s*$"
)


def _clean_value(raw: str) -> str:
    """Strip the optional [...] wrapper that DCMTK uses for some text values."""
    if raw.startswith("[") and raw.endswith("]"):
        return raw[1:-1]
    return raw


def parse_dump(text: str) -> Dataset:
    """Parse a DCMTK .dump file into a pydicom Dataset. Handles top-level
    elements + one level of sequence (enough for MWL fixtures)."""
    ds = Dataset()
    current_seq: str | None = None
    current_item: Dataset | None = None

    for raw in text.splitlines():
        line = raw.split("#", 1)[0].rstrip()
        if not line.strip():
            continue
        if "(fffe,e000)" in line.lower():
            current_item = Dataset()
            continue
        if "(fffe,e00d)" in line.lower():
            if current_item is not None and current_seq is not None:
                seq = list(getattr(ds, current_seq, []))
                seq.append(current_item)
                setattr(ds, current_seq, seq)
                current_item = None
            continue
        if "(fffe,e0dd)" in line.lower():
            current_seq = None
            current_item = None
            continue

        m = LINE_RE.match(line)
        if not m:
            continue
        group, element, vr, value = m.groups()
        tag = (int(group, 16), int(element, 16))
        value = _clean_value(value or "")

        if vr.upper() == "SQ":
            try:
                from pydicom.datadict import keyword_for_tag
                kw = keyword_for_tag(tag)
            except Exception:
                kw = None
            if kw:
                current_seq = kw
                setattr(ds, kw, [])
            continue

        try:
            target = current_item if current_item is not None else ds
            target.add_new(tag, vr, value if value else None)
        except Exception:
            # Skip elements we can't parse — better to keep the rest than fail.
            pass

    return ds


def dump_to_wl(dump_path: Path, wl_path: Path) -> Dataset:
    """Convert one .dump file to a binary .wl file. Returns the dataset."""
    ds = parse_dump(dump_path.read_text())

    # Ensure the file_meta block is valid for writing
    file_meta = FileMetaDataset()
    file_meta.MediaStorageSOPClassUID = "1.2.840.10008.5.1.4.31"  # MWL FIND
    file_meta.MediaStorageSOPInstanceUID = generate_uid()
    file_meta.TransferSyntaxUID = ExplicitVRLittleEndian
    file_meta.ImplementationClassUID = generate_uid()

    ds.file_meta = file_meta
    ds.is_little_endian = True
    ds.is_implicit_VR = False

    pydicom.dcmwrite(str(wl_path), ds, write_like_original=False)
    return ds


# Convert every .dump fixture to .wl
import shutil
shutil.rmtree(WL_DIR, ignore_errors=True)
WL_DIR.mkdir()

converted = []
failed = []
for dump in dump_files:
    out = WL_DIR / (dump.stem + ".wl")
    try:
        ds = dump_to_wl(dump, out)
        converted.append((dump.name, out, ds))
        print(f"  converted: {dump.name}  →  {out.name}  ({out.stat().st_size} bytes)")
    except Exception as e:
        failed.append((dump.name, str(e)))
        print(f"  FAILED:    {dump.name} — {e}")

print()
print(f"✓ Converted {len(converted)}/{len(dump_files)} fixtures")
if failed:
    print(f"  ({len(failed)} failures — these usually have nested sequences our simple parser doesn't handle)")

## 5. Read back one `.wl` file

Let's confirm the binary files we produced are valid DICOM and contain the priority tag CritCom needs.

In [ ]:
if converted:
    name, wl_path, _ = converted[0]
    ds = pydicom.dcmread(str(wl_path))
    print(f"=== {wl_path.name} (originally {name}) ===")
    print()
    print(f"Transfer syntax:                {ds.file_meta.TransferSyntaxUID}")
    print(f"SOP Class:                      {ds.file_meta.MediaStorageSOPClassUID}")
    print()
    for attr in [
        "PatientName", "PatientID", "PatientBirthDate", "PatientSex",
        "AccessionNumber", "RequestedProcedureID",
        "RequestedProcedureDescription", "RequestedProcedurePriority",
        "Modality", "StudyInstanceUID",
    ]:
        val = getattr(ds, attr, "(not set)")
        print(f"  {attr:32s} {val}")

## 6. Survey all fixtures

In [ ]:
import pandas as pd

rows = []
for f in sorted(WL_DIR.glob("*.wl")):
    try:
        ds = pydicom.dcmread(str(f))
        rows.append({
            "file": f.name,
            "patient_name": str(getattr(ds, "PatientName", "")).replace("^", " "),
            "patient_id": str(getattr(ds, "PatientID", "")),
            "accession": str(getattr(ds, "AccessionNumber", "")),
            "modality": str(getattr(ds, "Modality", "")),
            "priority": str(getattr(ds, "RequestedProcedurePriority", "")),
            "description": str(getattr(ds, "RequestedProcedureDescription", "")),
        })
    except Exception as e:
        rows.append({"file": f.name, "error": str(e)})

df = pd.DataFrame(rows)
df

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

if "priority" in df.columns:
    priorities = Counter(df["priority"].dropna())
    if priorities:
        axes[0].bar(priorities.keys(), priorities.values(), color="#BA7517", alpha=0.85)
        for x, y in priorities.items():
            axes[0].text(x, y + 0.05, str(y), ha="center")
    axes[0].set_xlabel("RequestedProcedurePriority")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Priority distribution (DCMTK fixtures)")

if "modality" in df.columns:
    modalities = Counter(df["modality"].dropna())
    if modalities:
        axes[1].bar(modalities.keys(), modalities.values(), color="#1D9E75", alpha=0.85)
        for x, y in modalities.items():
            axes[1].text(x, y + 0.05, str(y), ha="center")
    axes[1].set_xlabel("Modality")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Modality distribution")

plt.tight_layout()
plt.show()

## 7. Get the DCMTK commit SHA (for citation reproducibility)

For the paper, you'll want to cite the exact commit you pulled from. This makes the data reproducible.

In [ ]:
result = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=str(DCMTK_DIR), capture_output=True, text=True,
)
sha = result.stdout.strip()
print(f"DCMTK commit SHA: {sha}")
print(f"\nCite as:\n")
print(
    f"  DCMTK — OFFIS DICOM Toolkit. github.com/DCMTK/dcmtk @ {sha[:8]}\n"
    f"  Test fixtures from dcmwlm/data/wlistdb/OFFIS/.\n"
    f"  Eichelberg M, et al. Proc SPIE Medical Imaging. 2004;5371:57-68."
)

## 8. Integrate into CritCom

Once you're happy with the fixtures, copy them into the repo:

```bash
mkdir -p /path/to/agentic-ai-radiology/orthanc-worklists
cp dcmtk_workspace/worklists/*.wl /path/to/agentic-ai-radiology/orthanc-worklists/
```

Orthanc will pick them up automatically via the `./orthanc-worklists` volume mount in `docker-compose.yml`.

Test from CritCom:

```python
from critcom.tools.fetch_report_dicom import run
import asyncio

# DCMTK fixtures use various accession numbers — check the table above
# for what's available, then query by one of them
result = asyncio.run(run({"patient_name": ""}))   # empty match returns all
print(result)
```

---

## What changed vs. the earlier "dcm4che" notebook

| Earlier notebook | This notebook |
|---|---|
| Made-up patient names | Real DCMTK test fixtures |
| Synthetic IDs I invented | Actual fixtures shipped with DCMTK |
| Citation was misleading (dcm4che, but data was made up) | Citation is honest (DCMTK, real data from their repo) |
| 3 hand-built samples | All fixtures shipped in `dcmwlm/data/wlistdb/OFFIS/` |

The earlier `seed_dicom_dcm4che.py` script in your repo can be **kept as-is** for offline / no-network test runs (it generates synthetic samples without needing a clone), but the citable fixtures from this notebook should be the primary worklist data for the paper.
